# AURA fashion language model training pipeline

**Run in Google Colab:** open `https://colab.research.google.com/`, choose **File → Upload notebook**, upload this file, then choose **Runtime → Change runtime type → T4 GPU**. Run **Runtime → Run all** and approve the Google Drive mount prompt. The notebook uses only the existing Drive datasets and local random-initialized scratch model; it downloads no pretrained weights and generates no training data. Test files are read only in the final evaluation cell after both training stages and final model saving.

Checkpoints and metrics are saved to `/content/drive/MyDrive/AURA`. If Colab disconnects, rerun from the top; each stage resumes from its latest Drive checkpoint. Decreasing loss alone does not establish fashion reasoning quality.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE=Path('/content/drive/MyDrive/AURA')
MODEL_DIR=BASE/'models/aura_fashion_gpt_scratch'
KNOWLEDGE_TRAIN=BASE/'datasets/processed/final_corpus_deduplicated/train.jsonl'
KNOWLEDGE_VAL=BASE/'datasets/processed/final_corpus_deduplicated/validation.jsonl'
KNOWLEDGE_TEST=BASE/'datasets/processed/final_corpus_deduplicated/test.jsonl'
INSTR_TRAIN=BASE/'datasets/styling_examples/constraint_v2/aura_train.jsonl'
INSTR_VAL=BASE/'datasets/styling_examples/constraint_v2/aura_validation.jsonl'
INSTR_TEST=BASE/'datasets/styling_examples/constraint_v2/aura_test.jsonl'
STAGE1_DIR=BASE/'checkpoints/aura_stage1_knowledge'; STAGE2_DIR=BASE/'checkpoints/aura_stage2_instruction'
FINAL_DIR=BASE/'models/aura_fashion_gpt_final'; LOG_DIR=BASE/'training_logs'
required=[MODEL_DIR,KNOWLEDGE_TRAIN,KNOWLEDGE_VAL,KNOWLEDGE_TEST,INSTR_TRAIN,INSTR_VAL,INSTR_TEST]
missing=[str(p) for p in required if not p.exists()]
assert not missing, 'Missing Drive paths:\n'+'\n'.join(missing)
assert Path('/content/drive/MyDrive').exists(), 'Google Drive mount not detected.'
for p in required[1:]: print(f'{p}: {p.stat().st_size:,} bytes')


## Dependencies and local model compatibility
Installs a stable Transformers 4.x API. Model loading is local-only. Stop on tokenizer/config mismatch before training.

In [ ]:
%pip -q install "transformers>=4.44,<5" "datasets>=2.20,<4" "accelerate>=0.33,<2"
import torch, transformers, datasets, time, json, re, inspect, math
from transformers import AutoConfig,AutoTokenizer,AutoModelForCausalLM,Trainer,TrainingArguments,default_data_collator
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset
assert torch.cuda.is_available(), 'Select a GPU runtime.'
print(torch.cuda.get_device_name(0)); assert 'T4' in torch.cuda.get_device_name(0), 'Expected Tesla T4.'
tokenizer=AutoTokenizer.from_pretrained(str(MODEL_DIR),local_files_only=True,use_fast=True)
config=AutoConfig.from_pretrained(str(MODEL_DIR),local_files_only=True)
model=AutoModelForCausalLM.from_pretrained(str(MODEL_DIR),local_files_only=True)
print('torch',torch.__version__,'transformers',transformers.__version__,'vocab',len(tokenizer),config.vocab_size)
assert len(tokenizer)==config.vocab_size==8000
assert model.get_input_embeddings().weight.shape[0]==len(tokenizer)
assert (config.n_layer,config.n_embd,config.n_head)==(4,256,4)
assert sum(param.numel() for param in model.parameters())==5_240_320, 'Unexpected scratch parameter count.'
assert tokenizer.eos_token_id is not None, 'Tokenizer requires EOS token.'
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
model.config.pad_token_id=tokenizer.pad_token_id
model.config.use_cache=False  # saves T4 memory while gradient checkpointing is enabled
model=model.to('cuda')
model.eval()
probe=tokenizer('AURA fashion model compatibility check',return_tensors='pt').to('cuda')
with torch.no_grad(): probe_logits=model(**probe).logits
assert probe_logits.shape[-1]==len(tokenizer), 'Tokenizer/model forward API mismatch.'
print('Local tokenizer/model forward pass verified:',tuple(probe_logits.shape))


In [ ]:
# Verify record counts and fields for train/validation only. Test paths are existence-checked above, not opened.
def check_jsonl(path, fields):
 n=0
 with open(path,encoding='utf-8') as f:
  for line in f:
   if line.strip():
    row=json.loads(line); assert isinstance(row,dict) and set(fields)<=row.keys(), (path,row.keys()); n+=1
 return n
counts={p.name:check_jsonl(p,keys) for p,keys in [(KNOWLEDGE_TRAIN,['source','type','text']),(KNOWLEDGE_VAL,['source','type','text']),(INSTR_TRAIN,['id','category','input','output']),(INSTR_VAL,['id','category','input','output'])]}
print('Record counts:',counts)
assert counts[KNOWLEDGE_TRAIN.name]==93410
assert counts[INSTR_TRAIN.name]==4000 and counts[INSTR_VAL.name]==500


## Stage 1: causal knowledge pretraining
Knowledge entries are serialized with source/type/text, concatenated, and split into fixed non-overlapping 512-token blocks. Validation uses only the knowledge validation split.

In [ ]:
from itertools import chain
BLOCK_SIZE=512
assert BLOCK_SIZE <= int(getattr(config,'n_positions',512)), 'Training block exceeds the model context window.'
knowledge=load_dataset('json',data_files={'train':str(KNOWLEDGE_TRAIN),'validation':str(KNOWLEDGE_VAL)})
def tok_knowledge(b):
 texts=[f"Source: {s}\nType: {t}\n{x}" for s,t,x in zip(b['source'],b['type'],b['text'])]
 encoded=tokenizer(texts,add_special_tokens=False,truncation=False)
 encoded['input_ids']=[ids+[tokenizer.eos_token_id] for ids in encoded['input_ids']]
 encoded['attention_mask']=[mask+[1] for mask in encoded['attention_mask']]
 return encoded
kt=knowledge.map(tok_knowledge,batched=True,remove_columns=knowledge['train'].column_names,desc='Tokenize knowledge')
def pack(b):
 merged={k:list(chain.from_iterable(b[k])) for k in b}; total=len(merged['input_ids'])//BLOCK_SIZE*BLOCK_SIZE
 out={k:[v[i:i+BLOCK_SIZE] for i in range(0,total,BLOCK_SIZE)] for k,v in merged.items()}; out['labels']=[x.copy() for x in out['input_ids']]; return out
kb=kt.map(pack,batched=True,desc='Pack fixed blocks'); print({k:len(v) for k,v in kb.items()}); assert len(kb['train']) and len(kb['validation'])


In [ ]:
LOG_DIR.mkdir(parents=True,exist_ok=True)
_training_argument_names=inspect.signature(TrainingArguments.__init__).parameters
_trainer_argument_names=inspect.signature(Trainer.__init__).parameters
trainer_tokenizer_kwargs=({'processing_class':tokenizer} if 'processing_class' in _trainer_argument_names else {'tokenizer':tokenizer})
def args(out,epochs,lr,ev,save,bs,acc):
 kw=dict(output_dir=str(out),num_train_epochs=epochs,learning_rate=lr,per_device_train_batch_size=bs,per_device_eval_batch_size=bs,gradient_accumulation_steps=acc,warmup_ratio=.03,weight_decay=.01,logging_steps=50,eval_steps=ev,save_strategy='steps',save_steps=save,save_total_limit=3,fp16=True,bf16=False,gradient_checkpointing=True,report_to='none',logging_dir=str(LOG_DIR),seed=42,data_seed=42,load_best_model_at_end=True,metric_for_best_model='eval_loss',greater_is_better=False,prediction_loss_only=True,dataloader_num_workers=2,save_safetensors=True)
 eval_key='eval_strategy' if 'eval_strategy' in _training_argument_names else 'evaluation_strategy'
 kw[eval_key]='steps'
 return TrainingArguments(**kw)
def run(trainer,out,name):
 out.mkdir(parents=True,exist_ok=True); last=get_last_checkpoint(str(out)); print(name,'resume:',last or 'fresh','save:',out)
 train_start=time.time(); result=trainer.train(resume_from_checkpoint=last or None); train_runtime=time.time()-train_start
 eval_start=time.time(); ev=trainer.evaluate(); eval_runtime=time.time()-eval_start
 metrics=dict(result.metrics); metrics.update({'runtime_seconds_this_run':train_runtime,'validation_runtime_seconds':eval_runtime,'final_eval_loss':ev.get('eval_loss'),'checkpoint_dir':str(out)})
 metrics_path=LOG_DIR/f'{name}_metrics.json'
 prior=json.load(open(metrics_path)) if metrics_path.exists() else {}
 metrics['cumulative_runtime_seconds']=prior.get('cumulative_runtime_seconds',0)+train_runtime
 metrics['cumulative_validation_runtime_seconds']=prior.get('cumulative_validation_runtime_seconds',0)+eval_runtime
 with open(metrics_path,'w') as f: json.dump(metrics,f,indent=2)
 print(name,'train_loss',metrics.get('train_loss'),'validation_loss',ev.get('eval_loss'),'train_runtime_hours',round(train_runtime/3600,2),'cumulative_train_hours',round(metrics['cumulative_runtime_seconds']/3600,2),'checkpoints',out)
 return metrics
# Dataset labels are prepared explicitly; this collator preserves EOS targets.
collator=default_data_collator
t1=Trainer(model=model,args=args(STAGE1_DIR,3,5e-5,500,500,8,4),train_dataset=kb['train'],eval_dataset=kb['validation'],data_collator=collator,**trainer_tokenizer_kwargs)
m1=run(t1,STAGE1_DIR,'stage1_knowledge')


## Stage 2: response-only instruction fine-tuning
Only train and validation files are loaded. Prompt labels are `-100`, and answer plus EOS labels are retained. The test split remains untouched.

In [ ]:
instr=load_dataset('json',data_files={'train':str(INSTR_TRAIN),'validation':str(INSTR_VAL)})
MAX_LEN=512
def make_prompt(category,user_input): return f"### Category: {category}\n### User:\n{user_input}\n### Assistant:\n"
def encode(b):
 ids_all=[]; masks=[]; attn=[]
 for cat,inp,out in zip(b['category'],b['input'],b['output']):
  p=tokenizer(make_prompt(cat,inp),add_special_tokens=False)['input_ids']; a=tokenizer(str(out),add_special_tokens=False)['input_ids']
  a=a[:MAX_LEN-1]+[tokenizer.eos_token_id]; keep=max(0,MAX_LEN-len(a))
  if len(p)>keep:
   prefix=tokenizer(f"### Category: {cat}\n### User:\n",add_special_tokens=False)['input_ids']
   p=(prefix+p[-(keep-len(prefix)):]) if keep>len(prefix) else (p[-keep:] if keep else [])
  ids=p+a; labels=[-100]*len(p)+a.copy(); n=len(ids)
  ids_all.append(ids+[tokenizer.pad_token_id]*(MAX_LEN-n)); masks.append(labels+[-100]*(MAX_LEN-n)); attn.append([1]*n+[0]*(MAX_LEN-n))
 return {'input_ids':ids_all,'attention_mask':attn,'labels':masks}
it=instr.map(encode,batched=True,remove_columns=instr['train'].column_names,desc='Mask prompt tokens')
ex=it['train'][0]; positions=[i for i,x in enumerate(ex['labels']) if x!=-100]; assert positions and all(x==-100 for x in ex['labels'][:positions[0]])
print('instruction train/validation rows',len(it['train']),len(it['validation']),'prompt mask verified')
t2=Trainer(model=model,args=args(STAGE2_DIR,5,2e-5,100,100,4,4),train_dataset=it['train'],eval_dataset=it['validation'],**trainer_tokenizer_kwargs)
m2=run(t2,STAGE2_DIR,'stage2_instruction')


In [ ]:
# Save separate final model/tokenizer, leaving scratch model untouched.
FINAL_DIR.mkdir(parents=True,exist_ok=True); t2.model.config.use_cache=True; t2.save_model(str(FINAL_DIR)); tokenizer.save_pretrained(str(FINAL_DIR))
print('Final model:',FINAL_DIR,'Stage 1 checkpoints:',STAGE1_DIR,'Stage 2 checkpoints:',STAGE2_DIR,'metrics:',LOG_DIR)


## Final generation smoke check
Uses an illustrative wardrobe, not a test example. Checks parseable JSON with the requested keys and rejects item IDs outside the supplied wardrobe. This is a smoke check, not a quality evaluation.

In [ ]:
wardrobe=[{'item_id':'top_014','name':'cream linen shirt','type':'top'},{'item_id':'bottom_022','name':'navy trousers','type':'bottom'},{'item_id':'shoe_008','name':'brown loafers','type':'shoes'},{'item_id':'bag_003','name':'tan tote','type':'bag'}]
allowed={x['item_id'] for x in wardrobe}
query={'occasion':'smart casual office','weather':'mild','wardrobe':wardrobe,'request':'Recommend one outfit using only these items. Return JSON with keys outfit (array of objects with item_id), and rationale (string).'}
prompt=make_prompt('wardrobe_recommendation',json.dumps(query))
ctx=int(getattr(t2.model.config,'n_positions',512)); smoke_new=min(160,max(32,ctx//3)); smoke_input=max(32,ctx-smoke_new)
x=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=smoke_input).to(t2.model.device); t2.model.eval()
with torch.no_grad(): y=t2.model.generate(**x,max_new_tokens=smoke_new,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
raw=tokenizer.decode(y[0][x['input_ids'].shape[1]:],skip_special_tokens=True).strip(); print(raw)
def parse_object(s):
 s=re.sub(r'^```(?:json)?\s*|\s*```$','',s.strip(),flags=re.I); start=s.find('{'); assert start>=0,'No JSON object'
 depth=0; quoted=False; esc=False
 for i,c in enumerate(s[start:],start):
  if quoted:
   if esc: esc=False
   elif c=='\\': esc=True
   elif c=='"': quoted=False
  elif c=='"': quoted=True
  elif c=='{': depth+=1
  elif c=='}':
   depth-=1
   if depth==0: return json.loads(s[start:i+1])
 raise ValueError('Incomplete JSON')
smoke={'json_valid':False,'expected_structure_valid':False,'wardrobe_ids_valid':False,'generated_ids':[],'invalid_ids':[]}
try:
 obj=parse_object(raw); smoke['json_valid']=isinstance(obj,dict)
 smoke['expected_structure_valid']=smoke['json_valid'] and {'outfit','rationale'}<=obj.keys() and isinstance(obj['outfit'],list) and isinstance(obj['rationale'],str)
 if smoke['expected_structure_valid']:
  for item in obj['outfit']:
   if not isinstance(item,dict) or 'item_id' not in item: raise ValueError('Outfit entries need item_id')
   smoke['generated_ids'].append(str(item['item_id']))
  smoke['invalid_ids']=sorted(set(smoke['generated_ids'])-allowed)
  smoke['wardrobe_ids_valid']=bool(smoke['generated_ids']) and not smoke['invalid_ids'] and len(smoke['generated_ids'])==len(set(smoke['generated_ids']))
except Exception as e:
 smoke['error']=repr(e)
print('SMOKE VALIDATION:',json.dumps(smoke,indent=2))


Metrics are written to `/content/drive/MyDrive/AURA/training_logs`. Runtime is reported per invocation; resumed runs should be combined from their logs/checkpoint history for total wall time. Validation loss does not prove stylistic quality, constraint adherence, or user satisfaction. The test split is reserved for a separately planned final evaluation.

## Final held-out evaluation (run only after all training and model selection)
This is the first cell that reads either test split. It does not train, tune, or select checkpoints. It reports actual measurements for the saved model.

In [ ]:
# Final-only: no training, tuning, or checkpoint selection occurs after reading test files.
knowledge_test=load_dataset('json',data_files={'test':str(KNOWLEDGE_TEST)})['test']
ktest=knowledge_test.map(tok_knowledge,batched=True,remove_columns=knowledge_test.column_names,desc='Final knowledge test tokenization')
kblocks_test=ktest.map(pack,batched=True,desc='Pack held-out knowledge blocks')
from transformers import AutoModelForCausalLM
stage1_best=t1.state.best_model_checkpoint or get_last_checkpoint(str(STAGE1_DIR))
assert stage1_best, 'No Stage 1 checkpoint found for final test evaluation.'
knowledge_model=AutoModelForCausalLM.from_pretrained(stage1_best,local_files_only=True)
knowledge_eval=Trainer(model=knowledge_model,args=TrainingArguments(output_dir=str(BASE/'tmp_eval_knowledge'),per_device_eval_batch_size=8,fp16=True,report_to='none',prediction_loss_only=True),data_collator=collator,**trainer_tokenizer_kwargs)
k_test_metrics=knowledge_eval.evaluate(kblocks_test['test'])
k_test_metrics['perplexity']=math.exp(k_test_metrics['eval_loss']) if k_test_metrics.get('eval_loss',100)<50 else None

instruction_test=load_dataset('json',data_files={'test':str(INSTR_TEST)})['test']
itest=instruction_test.map(encode,batched=True,remove_columns=instruction_test.column_names,desc='Final instruction test tokenization')
i_test_metrics=t2.evaluate(eval_dataset=itest)
final_model=AutoModelForCausalLM.from_pretrained(str(FINAL_DIR),local_files_only=True).to('cuda').eval()
context_limit=int(getattr(final_model.config,'n_positions',512)); max_new_tokens=min(160,max(32,context_limit//3)); input_limit=max(32,context_limit-max_new_tokens)
json_valid=0; schema_matches=0; schema_comparable=0; id_grounded=0; id_evaluable=0; generated_count=0; generation_errors=[]
item_id_pattern=re.compile(r"""(?i)["'](?:item_id|itemId)["']\s*:\s*["']([^"']+)["']""")
def output_ids(value):
 found=[]
 def walk(obj):
  if isinstance(obj,dict):
   for key,val in obj.items():
    if key in ('item_id','itemId') and isinstance(val,(str,int)): found.append(str(val))
    elif key in ('item_ids','itemIds') and isinstance(val,list): found.extend(str(x) for x in val if isinstance(x,(str,int)))
    else: walk(val)
  elif isinstance(obj,list):
   for val in obj: walk(val)
 walk(value); return found
for row in instruction_test:
 prompt=make_prompt(row['category'],row['input'])
 enc=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=input_limit).to('cuda')
 with torch.no_grad(): out=final_model.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
 text=tokenizer.decode(out[0][enc['input_ids'].shape[1]:],skip_special_tokens=True).strip(); generated_count+=1
 try:
  obj=parse_object(text); json_valid+=1
  try:
   expected=parse_object(str(row['output'])); schema_comparable+=1
   if set(obj.keys())==set(expected.keys()): schema_matches+=1
  except Exception: pass
  supplied_ids=set(item_id_pattern.findall(str(row['input'])))
  predicted_ids=output_ids(obj)
  if supplied_ids:
   id_evaluable+=1
   if predicted_ids and set(predicted_ids)<=supplied_ids and len(predicted_ids)==len(set(predicted_ids)): id_grounded+=1
 except Exception as e:
  if len(generation_errors)<5: generation_errors.append({'id':row.get('id'),'error':str(e)[:160]})
instr_metrics={'examples':generated_count,'json_valid_count':json_valid,'json_valid_rate':json_valid/max(1,generated_count),'target_json_schema_comparable_count':schema_comparable,'expected_schema_match_count':schema_matches,'expected_schema_match_rate':schema_matches/schema_comparable if schema_comparable else None,'wardrobe_id_evaluable_examples':id_evaluable,'grounded_id_count':id_grounded,'grounded_id_rate':id_grounded/id_evaluable if id_evaluable else None,'parse_errors_sample':generation_errors}
final_report={'knowledge_test':k_test_metrics,'instruction_test':{'response_only_test_loss':i_test_metrics.get('eval_loss'),**instr_metrics},'test_used_for_training_or_selection':False,'model_dir':str(FINAL_DIR)}
with open(LOG_DIR/'final_test_evaluation.json','w') as f: json.dump(final_report,f,indent=2)
print(json.dumps(final_report,indent=2))
